# Petfinder Scraper Debugger (async)

Opens a Petfinder pet-detail URL in a **visible Chrome browser** via Playwright (same engine Apify uses).
Watch what loads, inspect the DOM, try carousel clicks, see exactly what's reachable to a scraper.

Uses Playwright's **async API** because Jupyter runs an asyncio loop already (sync API errors out).

**Use:** edit the URL in cell 3, run cells top-to-bottom. Browser stays open between cells until cell 9.

## 1. Install dependencies (one-time)

In [17]:
%pip install playwright beautifulsoup4
!playwright install chromium

Note: you may need to restart the kernel to use updated packages.


## 2. Pick a Petfinder URL

Replace with a pet detail URL from your last pipeline run.

In [40]:
TARGET_URL = "https://www.petfinder.com/cat/sebastian-860a0015-32ee-4755-bb07-e6022e1c8ef6/ga/marietta/good-mews-animal-foundation-ga173/details/"
# ↑ paste a real URL from your last Pets pipeline run

## 3. Open the page in a visible browser

`headless=False` makes Chrome visible. `slow_mo=300` slows actions so you can watch.

In [42]:
from playwright.async_api import async_playwright
import asyncio

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=300)
ctx = await browser.new_context(
    user_agent=("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"),
    viewport={"width": 1280, "height": 900},
)
page = await ctx.new_page()
await page.goto(TARGET_URL, wait_until="domcontentloaded")
print(f"Loaded: {page.url}")
print(f"Title:  {await page.title()}")

Loaded: https://www.petfinder.com/cat/sebastian-860a0015-32ee-4755-bb07-e6022e1c8ef6/ga/marietta/good-mews-animal-foundation-ga173/details/
Title:  Sebastian - Adoptable Pet | Petfinder | Petfinder


## 4. Wait for SPA data to render — try multiple selectors

Petfinder fetches the bio client-side after page load. Wait for any of these to appear.

In [44]:
selectors_to_try = [
    '[data-test="Pet_Story_Section"]',
    '[class*="description"]',
    '[class*="Description"]',
    '[class*="PetStory"]',
    '[class*="pet-story"]',
    'main p',
    'article',
]

for selector in selectors_to_try:
    try:
        el = await page.wait_for_selector(selector, timeout=4000)
        if el:
            text = (await el.inner_text())[:500]
            if text and len(text.strip()) > 30:
                print(f"✓ '{selector}' → {len(text)} chars")
                print(f"  Preview: {text[:200]}\n")
                continue
        print(f"·  '{selector}' — element exists but text < 30 chars")
    except Exception:
        print(f"✗ '{selector}' — not found within 4s")

await asyncio.sleep(2)

✗ '[data-test="Pet_Story_Section"]' — not found within 4s
✗ '[class*="description"]' — not found within 4s
✗ '[class*="Description"]' — not found within 4s
✗ '[class*="PetStory"]' — not found within 4s
✗ '[class*="pet-story"]' — not found within 4s
·  'main p' — element exists but text < 30 chars
✗ 'article' — not found within 4s


## 5. Inspect `__NEXT_DATA__` after the wait

In [47]:
import json

next_json = await page.evaluate("""() => {
    const tag = document.getElementById('__NEXT_DATA__');
    return tag ? tag.textContent : null;
}""")

if not next_json:
    print("No __NEXT_DATA__ tag found.")
else:
    nd = json.loads(next_json)
    pp = nd.get("props", {}).get("pageProps", {})
    print(f"pageProps top-level keys: {list(pp.keys())}")
    animal = pp.get("animal") or pp.get("pet") or {}
    if animal:
        print(f"\n✓ animal keys: {list(animal.keys())[:30]}")
        desc = animal.get("description", "") or ""
        print(f"✓ description length: {len(desc)}")
        print(f"  preview: {desc[:300]}")
        photos = animal.get("photos", [])
        print(f"✓ photos count: {len(photos)}")
        for i, ph in enumerate(photos[:3]):
            print(f"  photo {i}: {ph}")
    else:
        print("\n✗ No animal in pageProps — Petfinder didn't hydrate the data.")

pageProps top-level keys: ['initialApolloState', 'menuData', 'animal', 'animalMedia', 'breeds', 'drupalBreeds', 'organization', 'organizationContact', 'animalUrl', 'error', 'envVars']

✓ animal keys: ['animalId', 'animalName', 'animalType', 'description', 'extendedDescription', 'internalNotes', 'microchipId', 'primaryPhotoId', 'notes', 'tags', 'importUpdatesEnabled', 'importDeletesEnabled', 'matchLabel', 'distance', 'outOfTown', 'organization', '_organization', '_contact', '_location', 'physical', '_media', 'residency', 'behavior', 'publicUrl', 'sponsorAPetUrl', '__typename']
✓ description length: 990
  preview: Ben and Sebastian came to Good Mews from a county shelter for a better chance at adoption, and these two brothers have been side by side ever since.

Ben is the calmer of the pair and tends to take a little time to settle in, but once he feels comfortable, he really enjoys attention and gentle pets.
✓ photos count: 0


## 6. Find and click the carousel

In [49]:
carousel_button_candidates = [
    'button[aria-label="Next"]',
    'button[aria-label*="next" i]',
    '[data-testid="next"]',
    '[class*="next" i]',
    'svg[class*="chevron-right"]',
]

all_photos = set()

async def collect_visible_photos():
    urls = await page.evaluate("""() => {
        return Array.from(document.querySelectorAll('img'))
            .map(el => el.src || el.dataset.src || '')
            .filter(s => s && /\\.(jpg|jpeg|png|webp)/i.test(s))
            .filter(s => !/(logo|favicon|sprite|icon-|avatar)/i.test(s));
    }""")
    for u in urls:
        all_photos.add(u)

await collect_visible_photos()
print(f"After initial load: {len(all_photos)} unique photo URLs")

next_btn = None
for sel in carousel_button_candidates:
    btn = page.locator(sel).first
    if await btn.count() > 0:
        print(f"Found next button via: {sel}")
        next_btn = btn
        break

if next_btn:
    for i in range(4):
        try:
            await next_btn.click(timeout=2000)
            await asyncio.sleep(0.7)
            await collect_visible_photos()
            print(f"  Click {i+1} → {len(all_photos)} unique photos so far")
        except Exception as e:
            print(f"  Click {i+1} failed: {e}")
            break
else:
    print("No carousel-next button found.")

print(f"\nFinal unique photos: {len(all_photos)}")
for u in list(all_photos)[:10]:
    print(f"  {u}")

After initial load: 7 unique photo URLs
Found next button via: button[aria-label*="next" i]
  Click 1 → 7 unique photos so far
  Click 2 → 7 unique photos so far
  Click 3 failed: Locator.click: Timeout 2000ms exceeded.
Call log:
  - waiting for locator("button[aria-label*=\"next\" i]").first


Final unique photos: 7
  https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022e1c8ef6/image/100bbcf8-0f40-4dc0-9f4a-90cc7f1a6d57.jpg?versionId=vZfnRdz20WMsObHFtYCEY.zkUDAf2oSS
  https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022e1c8ef6/image/31fb9d7f-f713-4d8d-bd02-db7b3927c343.jpg?versionId=gb39AmT_0MFNA.oVBnZIhgpgolB9fcHy
  https://dbw3zep4prcju.cloudfront.net/animal/591e13cf-c767-4dfa-8f4b-2e00eb325d1c/image/8be67da2-b382-4ae8-b812-362156d46b9f.jpg
  https://dbw3zep4prcju.cloudfront.net/animal/643892e6-9bf4-40e7-bf42-2a6e91ded233/image/12778b8f-942a-4018-ba31-9e5b456e231b.jpg
  https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022

## 7. Screenshot for the record

In [51]:
from pathlib import Path
out = Path('output') / 'petfinder_debug.png'
out.parent.mkdir(exist_ok=True, parents=True)
await page.screenshot(path=str(out), full_page=True)
print(f"Saved: {out.resolve()}")

Saved: /Users/couch2coders/Documents/GitHub/newsletters/Pets/Code/output/petfinder_debug.png


## 8. Dump full HTML (post-JS) for offline grep

In [53]:
html_out = Path('output') / 'petfinder_rendered.html'
html_out.parent.mkdir(exist_ok=True, parents=True)
html_text = await page.content()
html_out.write_text(html_text, encoding='utf-8')
print(f"Saved: {html_out.resolve()}  ({html_out.stat().st_size:,} bytes)")

for keyword in ['Pet_Story_Section', 'description', '"animal":', 'rdcpix', 'cloudfront', '"photos"']:
    count = html_text.lower().count(keyword.lower())
    print(f"  '{keyword}' appears {count}x in rendered HTML")

Saved: /Users/couch2coders/Documents/GitHub/newsletters/Pets/Code/output/petfinder_rendered.html  (453,058 bytes)
  'Pet_Story_Section' appears 0x in rendered HTML
  'description' appears 11x in rendered HTML
  '"animal":' appears 1x in rendered HTML
  'rdcpix' appears 0x in rendered HTML
  'cloudfront' appears 31x in rendered HTML
  '"photos"' appears 1x in rendered HTML


## 9. Close the browser

In [ ]:
await browser.close()
await p.stop()
print("Closed.")

## What to do with findings

- **Cell 5 shows `animal` populated:** Bio is in the JSON; add a `wait_for_selector` to Apify.
- **Cell 5 still empty after waiting:** Petfinder fetches via XHR — intercept that, or use the official API.
- **Cell 6 collects 5+ photos after clicks:** Carousel clicking works — port back to Apify with right timing.
- **Cell 8 shows high `description` count:** Update `parse_detail_html` selectors based on cell 4.